<a href="https://colab.research.google.com/github/pritika-v/Mistral-large_from-scratch/blob/main/MISTRAL_LARGE_FROM_SCRATCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# torch     -> the deep learning framework we use for all tensor operations
# tiktoken  -> OpenAI's fast tokenizer (same BPE approach  as SentencePiece). Original mistral large uses the SentencePiece Tokenizers
# datasets  -> HuggingFace library to easily download training text data
"""
The original Mistral models use SentencePiece tokenizers, specifically a Byte-Pair Encoding (BPE) tokenizer trained on Mistral's corpus
"""
!pip install torch tiktoken datasets

In [5]:
import torch                          # Main deep learning library
import torch.nn as nn                 # Neural network building blocks (Linear, Module, etc.)
import torch.nn.functional as F       # Functions like softmax, relu, etc.
import tiktoken                       # BPE Tokenizer
import math                           # For sqrt in attention formula
import numpy as np                    # For array operations
from dataclasses import dataclass     # Clean way to define config as a class
from typing import Optional           # For type hints

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")

# Check if GPU is available — training is faster on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.is_available()) #True
print(torch.cuda.get_device_name(0))#Tesla T4
print(f"Using device: {device}")

All imports successful!
PyTorch version: 2.11.0+cu128
True
Tesla T4
Using device: cuda


In [6]:
@dataclass # dataclass auto-generates __init__ construct so we don't have to write it
class MistralConfig: #simply a class whose job is to store model settings like no of layers, kv heads etc
    """
    All hyperparameters for our mini Mistral model.
    The real Mistral Large has:
        - vocab_size     = 32000
        - hidden_size    = 4096
        - num_layers     = 32
        - num_heads      = 32
        - num_kv_heads   = 8   (GQA: 4 query heads share 1 KV head)
        - window_size    = 4096
    We use a tiny version here so it trains in minutes.
    """

    vocab_size: int    = 50257    # Number of tokens the tokenizer knows (tiktoken GPT-2 vocab)
    hidden_size: int   = 256      # Size of every token's vector (real Mistral uses 4096)
    num_layers: int    = 4        # Number of stacked transformer blocks (real Mistral uses 32)
    num_heads: int     = 8        # Number of Query attention heads
    num_kv_heads: int  = 2        # Number of Key/Value heads (GQA: 4 Q heads share 1 KV head)
    head_dim: int      = 32       # Size of each attention head vector (hidden_size / num_heads)
    ffn_hidden: int    = 512      # Hidden size inside the Feed Forward Network
    max_seq_len: int   = 512      # Maximum number of tokens the model can see at once
    window_size: int   = 64       # Sliding Window Attention: each token only looks back 64 tokens
    dropout: float     = 0.1      # Dropout rate: randomly zero out 10% of activations during training
    rope_theta: float  = 10000.0  # Base frequency for RoPE (controls how fast rotations change)

# Create the config object — we'll pass this to every model component
config = MistralConfig()
print("Config created!")
print(f"  Vocabulary size : {config.vocab_size}")
print(f"  Hidden size     : {config.hidden_size}")
print(f"  Layers          : {config.num_layers}")
print(f"  Attention heads : {config.num_heads} Q heads, {config.num_kv_heads} KV heads (GQA)")
print(f"  Window size     : {config.window_size} tokens (Sliding Window Attention)")

Config created!
  Vocabulary size : 50257
  Hidden size     : 256
  Layers          : 4
  Attention heads : 8 Q heads, 2 KV heads (GQA)
  Window size     : 64 tokens (Sliding Window Attention)


TOKENIZER TO CONVERT WoRDS TO NUMBERS/ TOKEN IDS (not embeddings). BPE TOKENIZER IS USED

In [7]:
tokenizer=tiktoken.get_encoding("gpt2") #loading the gpt2 tokenizer from tiktoken. It also used BPE method
sample_text=" I love pizza because it tastes good"
token_ids=tokenizer.encode(sample_text) # encode() converts text-> list of integer token IDs
print(f"Text : {sample_text}")
print(f"Tokens: {token_ids}")


Text :  I love pizza because it tastes good
Tokens: [314, 1842, 14256, 780, 340, 18221, 922]


In [8]:
#decode() converts text-->list of integer token IDs
decoded=tokenizer.decode(token_ids)
print(f"Decoded: {decoded}")


Decoded:  I love pizza because it tastes good


RMSNorm -NORMALIZING HIDDEN STATES

In [9]:
class RMSNorm(nn.Module):#Every neural network component in PyTorch inherits from nn.Module
    """
    This(nn.Module) tells PyTorch: "This class is a neural network layer."

    PyTorch then automatically tracks:

    weights
    gradients
    parameters

    for training.
    """
    """
    Root Mean Square Normalization.
    Keeps values in a stable range so gradients don't explode during training.
    Used in Mistral instead of regular LayerNorm because it's faster.

    Formula: x / sqrt(mean(x^2) + epsilon)  * weight
    """
    #hidden size = no of features in the input vector I-->[0.2, 0.1, 0.8] here hidden size is 3
    def __init__(self,hidden_size:int, eps:float=1e-6):
      super().__init__()
      self.eps=eps #stores 0.000001 . If mean_squared=0 then 1/sqrt(0) will give error so we add a small value of eps                                  # Small number to avoid dividing by zero
      self.weight=nn.Parameter(torch.ones(hidden_size)) # Learnable scale parameter
        #hidden size here is 256. torch.ones(256) gives [1,1,1,...] of size 256. nn.Parameter tells PyTorch "This tensor is trainable" (These weights will be updated during training)


    def forward(self, x:torch.Tensor)->torch.Tensor:
      # x shape: (batch_size, seq_len, hidden_size)-(2,10,256)
        #2 sequences, 10 tokens each, 256 features per token
      x_squared=x.pow(2)
      mean_squared=x_squared.mean(dim=-1, keepdim=True)# keepdim=True means we keep the same number of dimensions (so we can divide x by it later without shape issues)
        #mean(dim=-1) means we take the mean across hidden_size, so we get a single number for each token that represents the average of the squared values across all features.
      x_normed=x*torch.rsqrt(mean_squared+self.eps) # rsqrt = 1/sqrt
      return self.weight*x_normed #the real weight*normalized value. This allows the model to learn how much to scale the normalized output (if weight is 1, it keeps the same scale; if weight is 0.5, it halves it; if weight is 2, it doubles it, etc.)
#Quick Test
norm=RMSNorm(hidden_size=256)
dummy=torch.randn(2,10,256) #Generate random numbers from a normal distribution.We are doing this instead of getting sentences, converting to token
#I love pizza--> this sentence is 1 seuqnece. if there aren't 10 tokens in it, we pad.        # batch=2, seq_len=10, hidden=256
out=norm(dummy)
print(f"RMSNorm input  shape: {dummy.shape}")
print(f"RMSNorm output shape: {out.shape}")
print("RMSNorm works! ✅")



RMSNorm input  shape: torch.Size([2, 10, 256])
RMSNorm output shape: torch.Size([2, 10, 256])
RMSNorm works! ✅


RoPE — Rotary Positional Embedding (This is how Mistral tells each tokem where it is in the sequence. Instead of adding position numbers, it rotates the Q and K vectors)

In [10]:
class RotaryEmbedding(nn.Module):
  """
    RoPE: Rotary Positional Embedding.

    Core idea from your notes:
      - Old PE -> add position number to word embedding (absolute position)
      - RoPE   -> rotate Q and K vectors based on position (relative position)

    This makes the attention mechanism understand relative distances
    ("word B is 3 positions away from word A") rather than absolute indices.

    Uses sine and cosine rotations. Rotation angle = position × frequency.
    Applied INSIDE self-attention to Q and K vectors (not to embeddings).
    """
  def __init__(self,head_dim:int, rope_theta: float=10000.0):
    super().__init__()
    self.head_dim=head_dim #head_dim is the size of each attention head vector (hidden_size / num_heads). This is how many features each head processes. For example, if hidden_size=256 and num_heads=8, then head_dim=32.
    self.rope_theta=rope_theta #already given as 10,000
    inv_freq=1.0/(rope_theta**(torch.arange(0,head_dim,2).float()/head_dim))
    # Creates the frequency values for rotations(each dimension pair)
    #torch.arrange(0,head-dim(8),2) will give [0,2,4,6] which means we take every even index up to head_dim. We give 2 step because RoPE works on pairs of dimensionsThis is because each pair of dimensions (i and i+1) share the same frequency in RoPE. So if head_dim=32, we get 16 frequencies for the 16 pairs of dimensions.
    # divide them by head_dim [0,2,4,6] / 8 gives [0,0.25,0.5,0.75].
    #Then raise 10,000 to these powers
    #Then take reciprocal 1/frequency. Thes ebecome different frequencies that control how fast the rotation changes across dimensions. Lower frequencies change more slowly (good for long-range dependencies), higher frequencies change faster (good for short-range dependencies).
    self.register_buffer("inv_freq",inv_freq)
    # Register as a buffer (not a learnable/Trainable parameter like nn.Parameter, but saved with model)


  def _compute_cos_sin(self, seq_len: int, device:torch.device):#This creates rotation values.
    """Compute the cosine and sine rotation matrices for positions 0..seq_len-1."""
    positions=torch.arange(seq_len, device=device).float()
    # Create token posiitons : [0, 1, 2, ..., seq_len-1]

    freqs=torch.outer(positions, self.inv_freq)
    # Outer product: each position × each frequency

    emb=torch.cat([freqs,freqs],dim=-1)
    #RoPE rottes pairs of dimensions, so we duplicate frequencies [f1,f2,f3,f1,f2,f3]

    return emb.cos(), emb.sin() #These values are later used to rotate vectors.
    # Return cos and sin of the angles — these are what we use to rotate vectors

  def _rotate_half(self, x: torch.Tensor)->torch.Tensor:
    """Split vector in half, negate second half and swap. This is the rotation trick."""
    half=x.shape[-1]//2
    x1=x[...,:half] #First Half
    x2=x[..., half:] #second half
    return torch.cat([-x2,x1], dim=-1) #Rotate: [-x2, x1]

  def forward(self, q:torch.Tensor, k:torch.Tensor)->tuple:
    """
        Apply RoPE to both Q and K vectors.
        q, k shape: (batch, num_heads, seq_len, head_dim)
    """
    seq_len=q.shape[2] # How many tokens in this sequence

    # Get the cosine and sine rotation values for all positions
    cos,sin=self ._compute_cos_sin(seq_len, q.device)

    # Reshape for broadcasting: (1, 1, seq_len, head_dim)
    cos=cos.unsqueeze(0).unsqueeze(0)
    sin=sin.unsqueeze(0).unsqueeze(0)

    # Apply rotation to Q: q * cos + rotate(q) * sin
    q_rotated=(q*cos)+(self._rotate_half(q)*sin)
    #Apply same rotation to K
    k_rotated=(k*cos)+(self._rotate_half(k)*sin)

    return q_rotated, k_rotated

#----Quick Test----
rope=RotaryEmbedding(head_dim=32)
q_test=torch.randn(2,8,10,32) # batch=2, heads=8, seq_len=10, head_dim=32
k_test=torch.randn(2,8,10,32)
q_rot,k_rot=rope(q_test, k_test)
print(f"RoPE input  Q shape: {q_test.shape}")
print(f"RoPE output Q shape: {q_rot.shape}")
print("RoPE works! ✅")


RoPE input  Q shape: torch.Size([2, 8, 10, 32])
RoPE output Q shape: torch.Size([2, 8, 10, 32])
RoPE works! ✅


Step 7: Grouped Query Attention with Sliding Window

This is the most important part. Mistral's attention has two special properties:
1. **GQA** — multiple Q heads share fewer KV heads (saves memory)
2. **SWA** — each token only attends to the last `window_size` tokens (saves compute)

In [11]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention (GQA) + Sliding Window Attention (SWA).

    GQA from your notes:
      - Normal MHA: 32 Q heads, 32 K heads, 32 V heads -> huge KV cache
      - MQA (Multi-Query): all heads share 1 KV -> quality drops
      - GQA: groups of Q heads share 1 KV -> balance of speed & quality

    SWA from your notes:
      - Normal: each token looks at ALL previous tokens -> O(N^2)
      - SWA: each token only looks at last `window_size` tokens -> efficient
      - Long-range info propagates through stacked layers

    Formula: Attention(Q,K,V) = Softmax(Q @ K^T / sqrt(head_dim)) @ V
    Attention(Q, K, V) = Softmax(QKᵀ / √d_k) × V
    """

    def __init__(self, config: MistralConfig):
        super().__init__()
        self.num_heads    = config.num_heads      # let num_heads =8
        self.num_kv_heads = config.num_kv_heads   # 2, meaning key=2 and value=2
        self.head_dim     = config.head_dim        # head_dim=64 , each attention head stores 64 numbers
        self.window_size  = config.window_size     # windown_size=4 Each token only sees last 4 tokens instead of all previous tokens.

        self.kv_groups = self.num_heads // self.num_kv_heads #8 / 2 = 4 Meaning 4 Query heads share 1 Key head

        # Total size = heads × head_dim
        q_size  = self.num_heads    * self.head_dim   # 8 × 64 = 512 So Query projection outputs 512 numbers
        kv_size = self.num_kv_heads * self.head_dim   # 2 × 64 =128

        # These are the "learned weight matrices" Wq, Wk, Wv from your notes
        self.q_proj  = nn.Linear(config.hidden_size, q_size,  bias=False)  # Wq Shape 512 × 512
        self.k_proj  = nn.Linear(config.hidden_size, kv_size, bias=False)  # Wk Shape 512 × 128
        self.v_proj  = nn.Linear(config.hidden_size, kv_size, bias=False)  # Wv Shape 512 × 128
        self.o_proj  = nn.Linear(q_size, config.hidden_size,  bias=False)  # Final Matrix Wo

        # RoPE to rotate Q and K for positional awareness
        self.rope = RotaryEmbedding(config.head_dim, config.rope_theta) #this roatest vectors instead of addings them for positions. it roates only Q and K not V

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
          """
          x shape: (batch_size, seq_len, hidden_size) (2,5,512)-- 2 sentences, each sentence has 5 tokens, every token is represented by a 512-dimensional embedding
          """
          batch, seq_len, _ = x.shape

          # --- Step 1: Project input to Q, K, V ---
          q = self.q_proj(x)   #Q = XWq (What does each token want to find) Input (2,5,512) Output (2,5,512)
          k = self.k_proj(x)   # # K = X @ Wk  — what does each token offer/contain as a key? (2,5,128)--> 2 KV heads 64 each
          v = self.v_proj(x)   # # V = X @ Wv  — what actual information does each token carry? (2,5,128)

          # Split into multiple heads, hidden_size=512, num_heads=8, head_dim=64
          q = q.view(batch, seq_len, self.num_heads,    self.head_dim).transpose(1, 2) #Currently (2,5,512) becomes (2,5,8,64) meaning every token now owns 8 heads the transpose (2,5,8,64)->(2,8,5,64) because pytorch expects (batch,heads,sequence,head_dim)
          k = k.view(batch, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2) #(2,2,5,64)
          v = v.view(batch, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2) #(2,2,5,64)
          # All now shape: (batch, num_heads, seq_len, head_dim)
          #Before: (batch, seq_len, 512) After:(batch, seq_len, 8, 64) Tranpose interchanges index 1 and 2
          #Finally (2,8,5,64)-- instead of a 512-dimensional vector, each token now has 8 smaller vectors(8 attention heads), each of size 64

          q, k = self.rope(q, k) #Every Query and Key vector is rotated according to position. Rope is not applied to V

          # --- Step 4: Expand K and V for GQA --- rigth now we have 8 query heads but only 2 kv heads. therefore need matching dimensions
          #repeat interleaves will make 1,2,3,4 share k1,v1 5,6,7,8 share k2,v2
          #k1,k2 becomes k1 k1 k1 k1 k2 k2 k2 k2, same for v
          k = k.repeat_interleave(self.kv_groups, dim=1)
          v = v.repeat_interleave(self.kv_groups, dim=1)

          # --- Step 5: Compute Attention Scores ---
          # scores = Q @ K^T / sqrt(head_dim)--> This gives us "how much should each token pay attention to each other token?"
          scale  = math.sqrt(self.head_dim) # scale=sqrt(64)=8 Scaling prevents the dot products from becoming too large, which would make the softmax overly peaked and harder to train.
          scores = torch.matmul(q, k.transpose(-2, -1)) / scale  #This is the attention score score(i,j) means How much should token i attend to token j? For 5 tokens each token compares against 5 tokens
          #q=(2,8,5,64) transpose k =(2,8,64,5) Matrix multiplication (matmul) gives (2,8,5,5) Every token compares itself with every other token.

          # --- Step 6: Apply Sliding Window Mask ---
          # Create a mask that prevents tokens from attending beyond window_size. Also applies causal masking (can't look at FUTURE tokens)
          swa_mask = self._make_sliding_window_mask(seq_len, x.device) #if window=2, token 5 can see 5,4,3
          scores = scores + swa_mask  # Blocked positions become becomes -inf

          # --- Step 7: Apply Softmax to get Attention Weights ---
          # Converts raw scores to probabilities that sum to 1
          weights = F.softmax(scores, dim=-1)  #Softmax converts [4 2 1]-->[0.84 0.11 0.05] Now they have become probabilities

          # --- Step 8: Weighted sum of Value vectors ---
          context = torch.matmul(weights, v)   # Softmax(QKᵀ / √dₖ) × V therefore cultiplying with V

          # --- Step 9: Merge all heads back together ---
          context = context.transpose(1, 2).contiguous()             # ((2,8,5,64)--> transposed to (2,5,8,64)
          context = context.view(batch, seq_len, -1)                 # (2,5,8,64)--> flattened to (2,5,512)

          # --- Step 10: Final output projection ---
          out = self.o_proj(context)  # Output = Context × Wo This mixes information across all attention heads into the final hidden representation.
          #Output (2,5,512) Exactly the same shape as the input, so it can be passed to the next Transformer block.

          return out

    def _make_sliding_window_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
          """
          Create a combined causal + sliding window mask.

          Causal mask (from your notes): a token can only look BACKWARDS, not at future tokens.
          This is what prevents cheating during training (the triangle-shaped mask).

          Sliding window: also block attention to tokens MORE than window_size steps ago.
          """
          # Create a (seq_len x seq_len) matrix of positions
          row_idx = torch.arange(seq_len, device=device).unsqueeze(1)  # seq_len=5 gives [[0 1 2 3 4]]
          col_idx = torch.arange(seq_len, device=device).unsqueeze(0)  # [[0 1 2 3 4]] Each column represents a key token that could be attended to.

          # Causal condition: can only attend to tokens at or before current position
          causal = row_idx >= col_idx
          """
          The above statement produces
          T F F F F
          T T F F F
          T T T F F
          T T T T F
          T T T T T A token cannot look into the future.
          """

          # Sliding window condition: can only attend within window_size tokens back
          window = (row_idx - col_idx) <= self.window_size #Suppose window=2 Then token 4 (0-based indexing) can only attend to tokens 2, 3, and 4, while older tokens fall outside the allowed window.

          # Both conditions must be true to allow attention
          allowed = causal & window #A position is valid only if: it is not in the future, and it lies within the last window_size tokens.

          # Convert boolean mask to ADDITIVE MASK:
          # Where True  -> 0.0    (allow attention, softmax sees the real score)
          # Where False -> -inf   (block attention, softmax gives ~0 weight)
          mask = torch.where(allowed, torch.zeros_like(allowed, dtype=torch.float),
                                      torch.full_like(allowed, float('-inf'), dtype=torch.float))
          """
          produces a matrix like
          0      -inf  -inf  -inf  -inf
          0       0    -inf  -inf  -inf
          0       0     0    -inf  -inf
          -inf    0     0      0    -inf
          -inf   -inf   0      0      0
          """

          # Final Reshape for broadcasting: (1, 1, seq_len, seq_len)
          return mask.unsqueeze(0).unsqueeze(0) #Transforms (5,5) into (1,1,5,5)


# --- Quick test ---
attn = GroupedQueryAttention(config)
x_test = torch.randn(2, 10, 256)   # batch=2, seq_len=10, hidden=256
out_test = attn(x_test)
print(f"Attention input  shape: {x_test.shape}")
print(f"Attention output shape: {out_test.shape}")
print("Grouped Query Attention works! ✅")




Attention input  shape: torch.Size([2, 10, 256])
Attention output shape: torch.Size([2, 10, 256])
Grouped Query Attention works! ✅


## Step 8: Feed Forward Network with SwiGLU

After attention, every token passes through a small neural network independently.
Mistral uses **SwiGLU activation** — a smarter alternative to plain ReLU.